# TempoRAL: Expert Distill with Real π0.5-DROID

**Goal**: Test whether the MetaController discovers subtask boundaries (β_t) from the **real π0.5-DROID pretrained model's** frozen residual stream, using **DROID droid_100** data.

**Pipeline**:
1. Build Gemma-300M action expert with **adaRMSNorm** (matching π0.5 architecture)
2. Load real pretrained weights from `gs://openpi-assets/checkpoints/pi05_droid`
3. Freeze expert → extract residual streams at layer 9 on DROID actions
4. Train MetaController (BiGRU + SwitchingUnit + Hypernetwork decoder)
5. Evaluate: does β_t discover subtask boundaries without supervision?

**Hardware**: A100 (40GB) recommended. T4 may work with careful memory management.

In [ ]:
# Download DROID droid_100 dataset and pi0.5-DROID checkpoint
!mkdir -p /dataset /checkpoint
!gsutil -m cp -r gs://gresearch/robotics/droid_100 /dataset/droid_100/
!gsutil -m cp -r gs://openpi-assets/checkpoints/pi05_droid /checkpoint/pi05_droid/

In [ ]:
# Install dependencies (JAX/flax are pre-installed on Colab)
# Pin orbax-checkpoint to a version compatible with openpi checkpoints
!pip install -q "orbax-checkpoint>=0.6.0,<0.10.0" torch numpy matplotlib scikit-learn
!pip install -q tensorflow tensorflow-datasets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import math, time, gc, os
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Configuration

Matches the real π0.5-DROID action expert (Gemma-300M with adaRMSNorm):
- Width: 1024, Depth: 18, MLP dim: 4096
- GQA: 8 attention heads, 1 KV head, head dim 256
- Action dim: 32 (padded from raw 8D DROID actions), horizon: 15

In [ ]:
@dataclass
class Config:
    # Gemma-300M action expert (exact match with pi0.5)
    width: int = 1024
    depth: int = 18
    mlp_dim: int = 4096
    num_heads: int = 8
    num_kv_heads: int = 1
    head_dim: int = 256

    # Action space (DROID: 7 joint positions + 1 gripper = 8D, padded to 32D)
    action_dim: int = 32
    raw_action_dim: int = 8
    action_horizon: int = 15

    # Residual stream extraction
    intervention_layer: int = 9  # mid-depth of 18 layers

    # MetaController
    n_z: int = 32
    rank: int = 32
    encoder_hidden: int = 128
    alpha: float = 0.05  # KL weight (rate-distortion tradeoff)

    # Expert Distill training
    distill_steps: int = 5000
    distill_lr: float = 1e-3
    distill_batch_size: int = 32

    # Data processing
    subsequence_len: int = 50  # timesteps per training subsequence

cfg = Config()
print(f"Expert: {cfg.width}d x {cfg.depth}L, intervention at layer {cfg.intervention_layer}")
print(f"Action: raw {cfg.raw_action_dim}D -> padded {cfg.action_dim}D, horizon {cfg.action_horizon}")
print(f"MetaController: n_z={cfg.n_z}, rank={cfg.rank}, alpha={cfg.alpha}")

## 2. π0.5 Action Expert Architecture

Key difference from standard Gemma: **adaRMSNorm** — each RMSNorm has a learned Dense projection that takes timestep conditioning and produces `(scale, shift, gate)` for adaptive normalization with gated residual connections.

```
adaRMSNorm(x, cond):
    normed = RMSNorm(x)
    scale, shift, gate = Dense(cond).chunk(3)
    return normed * (1 + scale) + shift, gate

GatedResidual(x, y, gate):
    return x + y * gate
```

In [ ]:
# === Pi0.5 Action Expert: Gemma-300M with adaRMSNorm ===

class AdaRMSNorm(nn.Module):
    """RMSNorm with adaptive modulation for pi0.5.
    With cond: normed * (1 + scale) + shift, returns gate for gated residual.
    Without cond: normed * (1 + weight), gate = None.
    """
    def __init__(self, dim, eps=1e-6, cond_dim=None):
        super().__init__()
        self.eps = eps
        self.dim = dim
        if cond_dim is not None:
            self.dense = nn.Linear(cond_dim, dim * 3, bias=True)
            nn.init.zeros_(self.dense.weight)
            nn.init.zeros_(self.dense.bias)
        else:
            self.weight = nn.Parameter(torch.zeros(dim))
            self.dense = None

    def _norm(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x, cond=None):
        normed = self._norm(x.float()).type_as(x)
        if cond is None or self.dense is None:
            return normed * (1.0 + self.weight), None
        modulation = self.dense(cond)
        # Broadcast: if x is (B,T,D) and cond is (B,D), unsqueeze modulation
        if len(x.shape) == 3 and len(modulation.shape) == 2:
            modulation = modulation.unsqueeze(1)
        scale, shift, gate = modulation.chunk(3, dim=-1)
        return normed * (1 + scale) + shift, gate


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_len=512):
        super().__init__()
        freqs = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_len).float()
        emb = torch.einsum("i,j->ij", t, freqs)
        self.register_buffer("cos", emb.cos())
        self.register_buffer("sin", emb.sin())

    def forward(self, x, offset=0):
        T = x.shape[-2]
        cos = self.cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class GQAttention(nn.Module):
    """Grouped-Query Attention (8 heads, 1 KV head for Gemma-300M)."""
    def __init__(self, width, num_heads, num_kv_heads, head_dim):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.kv_groups = num_heads // num_kv_heads
        self.q_proj = nn.Linear(width, num_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(width, num_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(width, num_kv_heads * head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * head_dim, width, bias=False)
        self.rope = RotaryEmbedding(head_dim)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        if self.kv_groups > 1:
            k = k.repeat_interleave(self.kv_groups, dim=1)
            v = v.repeat_interleave(self.kv_groups, dim=1)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.o_proj(attn.transpose(1, 2).contiguous().view(B, T, -1))


class GeGLU_FFN(nn.Module):
    def __init__(self, width, mlp_dim):
        super().__init__()
        self.gate_proj = nn.Linear(width, mlp_dim, bias=False)
        self.up_proj = nn.Linear(width, mlp_dim, bias=False)
        self.down_proj = nn.Linear(mlp_dim, width, bias=False)

    def forward(self, x):
        return self.down_proj(F.gelu(self.gate_proj(x), approximate='tanh') * self.up_proj(x))


class TransformerBlock(nn.Module):
    def __init__(self, width, num_heads, num_kv_heads, head_dim, mlp_dim, cond_dim):
        super().__init__()
        self.input_layernorm = AdaRMSNorm(width, cond_dim=cond_dim)
        self.self_attn = GQAttention(width, num_heads, num_kv_heads, head_dim)
        self.post_attention_layernorm = AdaRMSNorm(width, cond_dim=cond_dim)
        self.mlp = GeGLU_FFN(width, mlp_dim)

    def forward(self, x, cond=None):
        # Pre-attention norm (adaRMS) + gated residual
        normed, gate = self.input_layernorm(x, cond=cond)
        attn_out = self.self_attn(normed)
        x = x + attn_out * gate if gate is not None else x + attn_out
        # Pre-FFN norm (adaRMS) + gated residual
        normed, gate = self.post_attention_layernorm(x, cond=cond)
        ffn_out = self.mlp(normed)
        x = x + ffn_out * gate if gate is not None else x + ffn_out
        return x


class Pi05ActionExpert(nn.Module):
    """Pi0.5 Action Expert (Gemma-300M with adaRMSNorm).
    Exposes residual stream at intervention_layer for MetaController.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.action_in_proj = nn.Linear(cfg.action_dim, cfg.width)   # 32 -> 1024
        self.action_out_proj = nn.Linear(cfg.width, cfg.action_dim)  # 1024 -> 32
        self.time_mlp_in = nn.Linear(cfg.width, cfg.width)
        self.time_mlp_out = nn.Linear(cfg.width, cfg.width)
        self.layers = nn.ModuleList([
            TransformerBlock(cfg.width, cfg.num_heads, cfg.num_kv_heads,
                             cfg.head_dim, cfg.mlp_dim, cond_dim=cfg.width)
            for _ in range(cfg.depth)
        ])
        self.final_norm = AdaRMSNorm(cfg.width, cond_dim=cfg.width)
        self._residual_stream = None

    def _sinusoidal_embedding(self, timestep):
        """Sinusoidal embedding matching pi0.5 (min_period=4e-3, max_period=4.0)."""
        min_ts, max_ts = 4e-3, 4.0
        half = self.cfg.width // 2
        log_inc = math.log(max_ts / min_ts) / max(half - 1, 1)
        inv_ts = min_ts * torch.exp(
            torch.arange(half, device=timestep.device).float() * -log_inc
        )
        scaled = timestep.unsqueeze(-1) * inv_ts.unsqueeze(0)
        return torch.cat([scaled.sin(), scaled.cos()], dim=-1)

    def forward(self, actions, timestep=None, extract_residual=False):
        """Forward pass. actions: (B, T, 32), timestep: (B,) in [0,1]."""
        x = self.action_in_proj(actions)
        cond = None
        if timestep is not None:
            t_emb = self._sinusoidal_embedding(timestep)
            cond = F.silu(self.time_mlp_in(t_emb))
            cond = F.silu(self.time_mlp_out(cond))  # (B, width)
        self._residual_stream = None
        for i, layer in enumerate(self.layers):
            x = layer(x, cond=cond)
            if extract_residual and i == self.cfg.intervention_layer:
                self._residual_stream = x.detach()
        normed, _ = self.final_norm(x, cond=cond)
        return self.action_out_proj(normed)

    def get_residual_stream(self):
        assert self._residual_stream is not None, "Call forward with extract_residual=True"
        return self._residual_stream

expert = Pi05ActionExpert(cfg)
n_params = sum(p.numel() for p in expert.parameters())
print(f"Pi0.5 Action Expert: {n_params / 1e6:.1f}M parameters")
print(f"  (Gemma-300M base ~311M + adaRMSNorm ~115M + projections ~4M)")

## 3. Load Pretrained Weights from JAX Checkpoint

The pi0.5-DROID checkpoint is in orbax (JAX) format. We:
1. Load params with orbax + restore as numpy arrays
2. Extract only expert weights (suffix `_1` in key names)
3. Convert JAX array shapes to PyTorch Linear weight convention

**Key weight mappings**:
- `q_einsum_1[i]`: (num_heads, width, head_dim) → q_proj.weight (nh*hd, width)
- `kv_einsum_1[i, 0/1, 0]`: (width, head_dim) → k/v_proj.weight (head_dim, width)
- `gating_einsum[i, 0/1]`: (width, mlp_dim) → gate/up_proj.weight (mlp_dim, width)
- adaRMSNorm Dense_0 kernel: (cond_dim, dim*3) → dense.weight (dim*3, cond_dim)

In [ ]:
import orbax.checkpoint as ocp
import jax
from flax import traverse_util

def load_expert_params(checkpoint_dir):
    """Load JAX checkpoint params, compatible with multiple orbax versions."""
    params_path = f"{checkpoint_dir}/params/"
    print(f"Loading checkpoint from {params_path}...")

    # List contents for debugging
    import os
    if os.path.exists(params_path):
        contents = os.listdir(params_path)
        print(f"  Directory contents ({len(contents)} items): {contents[:10]}...")

    params = None

    # Strategy 1: Direct restore (works in many orbax versions)
    try:
        ckptr = ocp.PyTreeCheckpointer()
        raw = ckptr.restore(params_path)
        # Navigate to params if nested
        if isinstance(raw, dict) and "params" in raw:
            params = raw["params"]
        else:
            params = raw
        print("Loaded with PyTreeCheckpointer.restore() (direct)")
    except Exception as e1:
        print(f"Strategy 1 failed: {e1}")

        # Strategy 2: Use metadata + PyTreeRestore with proper API
        try:
            ckptr = ocp.PyTreeCheckpointer()
            metadata = ckptr.metadata(params_path)

            # Handle StepMetadata (newer orbax)
            if hasattr(metadata, 'item_metadata'):
                tree_meta = metadata.item_metadata
            elif isinstance(metadata, dict):
                tree_meta = metadata
            else:
                # Try to use metadata directly as tree structure
                tree_meta = metadata

            # Build restore args from metadata tree
            restore_args = jax.tree.map(
                lambda _: ocp.ArrayRestoreArgs(restore_type=np.ndarray),
                tree_meta,
            )
            raw = ckptr.restore(params_path, restore_args=restore_args)
            if isinstance(raw, dict) and "params" in raw:
                params = raw["params"]
            else:
                params = raw
            print("Loaded with PyTreeCheckpointer + metadata restore args")
        except Exception as e2:
            print(f"Strategy 2 failed: {e2}")

            # Strategy 3: Use StandardCheckpointer
            try:
                ckptr = ocp.StandardCheckpointer()
                raw = ckptr.restore(params_path)
                if isinstance(raw, dict) and "params" in raw:
                    params = raw["params"]
                else:
                    params = raw
                print("Loaded with StandardCheckpointer")
            except Exception as e3:
                print(f"Strategy 3 failed: {e3}")
                raise RuntimeError(
                    f"Could not load checkpoint from {params_path}. "
                    f"Try: pip install orbax-checkpoint==0.6.4\n"
                    f"Errors: {e1} / {e2} / {e3}"
                )

    # Convert jax arrays to numpy float32 if needed
    def to_numpy_f32(x):
        if isinstance(x, np.ndarray):
            return x.astype(np.float32) if x.dtype != np.float32 else x
        if hasattr(x, 'numpy'):  # tf tensor
            return np.array(x, dtype=np.float32)
        if hasattr(x, '__array__'):  # jax array
            return np.asarray(x, dtype=np.float32)
        return x

    params = jax.tree.map(to_numpy_f32, params)

    # Remove "value" suffix if present (nnx.State artifact)
    flat = traverse_util.flatten_dict(params)
    if flat and all(kp[-1] == "value" for kp in flat):
        print("Stripping 'value' suffix from keys...")
        flat = {kp[:-1]: v for kp, v in flat.items()}
    params = traverse_util.unflatten_dict(flat)

    print(f"Top-level keys: {list(params.keys())}")
    return params

params = load_expert_params("/checkpoint/pi05_droid")

In [ ]:
# Flatten PaliGemma params for easy access
def flatten_dict(d, parent_key='', sep='/'):
    items = {}
    for k, v in d.items():
        key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, key, sep))
        else:
            items[key] = v
    return items

pali_flat = flatten_dict(params["PaliGemma"])

# Show expert-related keys (suffix _1)
expert_keys = sorted([k for k in pali_flat if '_1' in k.split('/')[-1] or '_1/' in k])
print(f"Expert-related keys: {len(expert_keys)}")
for k in expert_keys[:10]:
    print(f"  {k}: shape={pali_flat[k].shape}")
print("  ...")

# Show projection keys
for name in ["action_in_proj", "action_out_proj", "time_mlp_in", "time_mlp_out"]:
    kernel = params[name]["kernel"]
    bias = params[name]["bias"]
    if isinstance(kernel, dict):
        kernel = kernel["value"]
        bias = bias["value"]
    print(f"{name}: kernel={kernel.shape}, bias={bias.shape}")

In [ ]:
def convert_expert_weights(pali_flat, params, cfg):
    """Convert JAX expert weights to PyTorch state dict."""
    sd = {}
    p = "llm/layers"  # JAX key prefix for transformer layers

    # Stacked arrays: shape[0] = num_layers
    q_einsum = pali_flat[f"{p}/attn/q_einsum_1/w"]
    kv_einsum = pali_flat[f"{p}/attn/kv_einsum_1/w"]
    attn_vec = pali_flat[f"{p}/attn/attn_vec_einsum_1/w"]
    gating = pali_flat[f"{p}/mlp_1/gating_einsum"]
    linear = pali_flat[f"{p}/mlp_1/linear"]
    in_norm_k = pali_flat[f"{p}/pre_attention_norm_1/Dense_0/kernel"]
    in_norm_b = pali_flat[f"{p}/pre_attention_norm_1/Dense_0/bias"]
    ff_norm_k = pali_flat[f"{p}/pre_ffw_norm_1/Dense_0/kernel"]
    ff_norm_b = pali_flat[f"{p}/pre_ffw_norm_1/Dense_0/bias"]

    print(f"q_einsum shape: {q_einsum.shape}")      # (18, 8, 1024, 256)
    print(f"kv_einsum shape: {kv_einsum.shape}")    # (18, 2, 1, 1024, 256)
    print(f"attn_vec shape: {attn_vec.shape}")      # (18, 8, 256, 1024)
    print(f"gating shape: {gating.shape}")          # (18, 2, 1024, 4096)
    print(f"linear shape: {linear.shape}")          # (18, 4096, 1024)
    print(f"in_norm_k shape: {in_norm_k.shape}")    # (18, 1024, 3072)

    nh, hd, w = cfg.num_heads, cfg.head_dim, cfg.width

    for i in range(cfg.depth):
        lp = f"layers.{i}"

        # Q: (num_heads, width, head_dim) -> (nh*hd, width)
        sd[f"{lp}.self_attn.q_proj.weight"] = torch.from_numpy(
            q_einsum[i].transpose(0, 2, 1).reshape(nh * hd, w).copy()
        )
        # K: kv_einsum[i, 0, 0] = (width, head_dim) -> transpose = (hd, width)
        sd[f"{lp}.self_attn.k_proj.weight"] = torch.from_numpy(
            kv_einsum[i, 0, 0].T.copy()
        )
        # V: kv_einsum[i, 1, 0]
        sd[f"{lp}.self_attn.v_proj.weight"] = torch.from_numpy(
            kv_einsum[i, 1, 0].T.copy()
        )
        # O: (num_heads, head_dim, width) -> reshape(nh*hd, width) -> transpose
        sd[f"{lp}.self_attn.o_proj.weight"] = torch.from_numpy(
            attn_vec[i].reshape(nh * hd, w).T.copy()
        )
        # FFN gate/up/down
        sd[f"{lp}.mlp.gate_proj.weight"] = torch.from_numpy(gating[i, 0].T.copy())
        sd[f"{lp}.mlp.up_proj.weight"] = torch.from_numpy(gating[i, 1].T.copy())
        sd[f"{lp}.mlp.down_proj.weight"] = torch.from_numpy(linear[i].T.copy())

        # adaRMSNorm: kernel.T -> dense.weight, bias -> dense.bias
        sd[f"{lp}.input_layernorm.dense.weight"] = torch.from_numpy(in_norm_k[i].T.copy())
        sd[f"{lp}.input_layernorm.dense.bias"] = torch.from_numpy(in_norm_b[i].copy())
        sd[f"{lp}.post_attention_layernorm.dense.weight"] = torch.from_numpy(ff_norm_k[i].T.copy())
        sd[f"{lp}.post_attention_layernorm.dense.bias"] = torch.from_numpy(ff_norm_b[i].copy())

    # Final norm (adaRMSNorm)
    fn_k = pali_flat["llm/final_norm_1/Dense_0/kernel"]
    fn_b = pali_flat["llm/final_norm_1/Dense_0/bias"]
    sd["final_norm.dense.weight"] = torch.from_numpy(fn_k.T.copy())
    sd["final_norm.dense.bias"] = torch.from_numpy(fn_b.copy())

    # Projection layers (top-level in params, not under PaliGemma)
    for name in ["action_in_proj", "action_out_proj", "time_mlp_in", "time_mlp_out"]:
        kernel = params[name]["kernel"]
        bias = params[name]["bias"]
        if isinstance(kernel, dict):
            kernel, bias = kernel["value"], bias["value"]
        sd[f"{name}.weight"] = torch.from_numpy(np.array(kernel).T.copy())
        sd[f"{name}.bias"] = torch.from_numpy(np.array(bias).copy())

    return sd

print("Converting JAX -> PyTorch weights...")
pt_state = convert_expert_weights(pali_flat, params, cfg)
print(f"Converted {len(pt_state)} weight tensors")

# Load into model
missing, unexpected = expert.load_state_dict(pt_state, strict=False)
if missing:
    print(f"WARNING - Missing keys ({len(missing)}): {missing[:5]}...")
if unexpected:
    print(f"WARNING - Unexpected keys ({len(unexpected)}): {unexpected[:5]}...")
if not missing and not unexpected:
    print("All weights loaded successfully!")

# Freeze and move to GPU
expert = expert.to(device).eval()
for p in expert.parameters():
    p.requires_grad = False

# Free JAX params
del params, pali_flat, pt_state
gc.collect()
print(f"\nPi0.5 Action Expert loaded and frozen on {device}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Load DROID Data

DROID droid_100: 100 robot manipulation episodes.
- Actions: 7 joint positions + 1 gripper = 8D
- Normalize with quantile clipping, then pad to 32D (matching pi0.5's action_dim)

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf

# Disable GPU for TF (we use PyTorch for GPU)
tf.config.set_visible_devices([], 'GPU')

# Load DROID droid_100
data_dir = "/dataset/droid_100"
try:
    builder = tfds.builder_from_directory(data_dir)
    ds = builder.as_dataset(split='train')
    print(f"Loaded DROID from builder_from_directory")
except Exception as e:
    print(f"builder_from_directory failed: {e}")
    print("Trying tfds.load...")
    ds = tfds.load('droid_100', data_dir=os.path.dirname(data_dir), split='train')

# Extract action sequences from episodes
episodes_raw = []
for episode in ds:
    actions = []
    for step in episode['steps']:
        act = step['action']
        # DROID actions: joint_position (7D) + gripper_position (1D)
        if isinstance(act, dict):
            joint = act.get('joint_position', act.get('world_vector', tf.zeros(7)))
            gripper = act.get('gripper_position', act.get('gripper_closedness_action', tf.zeros(1)))
            a = tf.concat([tf.reshape(joint, [-1]), tf.reshape(gripper, [-1])], axis=0)
        else:
            a = act  # already concatenated
        actions.append(a.numpy())
    if len(actions) > 10:  # skip very short episodes
        episodes_raw.append(np.array(actions, dtype=np.float32))

print(f"Loaded {len(episodes_raw)} episodes")
print(f"Episode lengths: min={min(len(e) for e in episodes_raw)}, "
      f"max={max(len(e) for e in episodes_raw)}, "
      f"mean={np.mean([len(e) for e in episodes_raw]):.0f}")
print(f"Action dim: {episodes_raw[0].shape[1]}")

In [ ]:
# Normalize actions with quantile clipping and pad to 32D
all_actions_raw = np.concatenate(episodes_raw, axis=0)
print(f"Total action frames: {len(all_actions_raw)}")

# Compute quantile normalization stats
q01 = np.percentile(all_actions_raw, 1, axis=0)
q99 = np.percentile(all_actions_raw, 99, axis=0)
print(f"q01: {q01}")
print(f"q99: {q99}")

def normalize_actions(actions, q01, q99):
    """Quantile normalization to [-1, 1]."""
    range_ = np.maximum(q99 - q01, 1e-6)
    normed = (actions - q01) / range_  # [0, 1]
    normed = normed * 2 - 1            # [-1, 1]
    return np.clip(normed, -1, 1)

def pad_to_action_dim(actions, target_dim=32):
    """Zero-pad from raw_dim to target_dim."""
    raw_dim = actions.shape[-1]
    if raw_dim >= target_dim:
        return actions[..., :target_dim]
    pad_width = [(0, 0)] * (actions.ndim - 1) + [(0, target_dim - raw_dim)]
    return np.pad(actions, pad_width, mode='constant', constant_values=0)

# Process episodes: normalize + pad
episodes = []
for ep in episodes_raw:
    normed = normalize_actions(ep, q01, q99)
    padded = pad_to_action_dim(normed, cfg.action_dim)
    episodes.append(padded.astype(np.float32))

print(f"\nProcessed {len(episodes)} episodes")
print(f"Action shape per step: {episodes[0].shape[1]} (padded from {episodes_raw[0].shape[1]})")

# Visualize first episode
fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
ep0 = episodes[0]
axes[0].plot(ep0[:, :7], alpha=0.7)
axes[0].set_ylabel('Joint positions')
axes[0].set_title(f'Episode 0 (normalized, {len(ep0)} steps)')
axes[0].legend([f'j{i}' for i in range(7)], fontsize=7, ncol=7)
axes[1].plot(ep0[:, 7], 'k-', linewidth=2)
axes[1].set_ylabel('Gripper')
axes[1].set_xlabel('Timestep')
plt.tight_layout()
plt.show()

## 5. Extract Residual Streams

Forward-pass DROID action sequences through the frozen expert using a sliding window of `action_horizon=15`. Extract residual stream at layer 9, then mean-pool over the 15 tokens to get a per-timestep vector of dim 1024.

In [ ]:
def extract_residual_streams(expert, episodes, cfg, batch_size=8):
    """Extract residual streams from frozen expert for all episodes.

    Uses sliding window of action_horizon over each episode.
    Mean-pools over tokens to get per-timestep residual vectors.
    """
    all_streams = []  # list of (T_i, width) arrays
    horizon = cfg.action_horizon

    with torch.no_grad():
        for ep_idx, ep in enumerate(episodes):
            T = len(ep)
            if T < horizon:
                continue

            # Sliding window: each step t uses actions[t:t+horizon]
            windows = []  # (num_windows, horizon, action_dim)
            for t in range(T - horizon + 1):
                windows.append(ep[t:t + horizon])
            windows = np.array(windows, dtype=np.float32)

            # Process in batches
            ep_streams = []
            for i in range(0, len(windows), batch_size):
                batch = torch.from_numpy(windows[i:i+batch_size]).to(device)
                t_val = torch.zeros(batch.shape[0], device=device)  # t=0 (clean)
                expert(batch, timestep=t_val, extract_residual=True)
                residual = expert.get_residual_stream()  # (B, horizon, width)
                # Mean-pool over tokens -> (B, width)
                pooled = residual.mean(dim=1).cpu().numpy()
                ep_streams.append(pooled)

            ep_streams = np.concatenate(ep_streams, axis=0)  # (T-horizon+1, width)
            all_streams.append(ep_streams)

            if (ep_idx + 1) % 10 == 0:
                print(f"  Processed {ep_idx+1}/{len(episodes)} episodes")

    return all_streams

print("Extracting residual streams from frozen pi0.5 expert...")
start = time.time()
residual_streams = extract_residual_streams(expert, episodes, cfg)
elapsed = time.time() - start

total_frames = sum(len(s) for s in residual_streams)
print(f"\nExtracted {len(residual_streams)} episode streams in {elapsed:.1f}s")
print(f"Total frames: {total_frames}")
print(f"Residual dim: {residual_streams[0].shape[1]}")
print(f"Memory: {total_frames * cfg.width * 4 / 1e6:.1f} MB")

## 6. Derive Approximate Boundaries

Since DROID has no ground-truth subtask labels, we derive approximate boundaries from:
1. **Action velocity**: large changes in joint positions indicate phase transitions
2. **Gripper state changes**: open/close transitions are strong boundary signals

These serve as evaluation references (not used in training).

In [ ]:
def derive_approximate_boundaries(episodes, cfg, vel_threshold=0.3, gripper_threshold=0.2):
    """Derive approximate subtask boundaries from action statistics."""
    horizon = cfg.action_horizon
    all_boundaries = []

    for ep in episodes:
        T = len(ep)
        if T < horizon:
            continue
        effective_T = T - horizon + 1  # matches residual_streams length

        # Use original (pre-padding) dimensions
        joints = ep[:effective_T, :7]   # joint positions
        gripper = ep[:effective_T, 7]   # gripper

        # Action velocity (L2 norm of joint differences)
        joint_vel = np.zeros(effective_T)
        joint_vel[1:] = np.linalg.norm(np.diff(joints, axis=0), axis=1)

        # Gripper state changes
        gripper_change = np.zeros(effective_T)
        gripper_change[1:] = np.abs(np.diff(gripper))

        # Combine signals
        vel_norm = joint_vel / (np.max(joint_vel) + 1e-8)
        grip_norm = gripper_change / (np.max(gripper_change) + 1e-8)

        boundary_score = 0.5 * vel_norm + 0.5 * grip_norm
        boundaries = (boundary_score > vel_threshold).astype(np.float32)

        # Enforce minimum gap between boundaries
        min_gap = 5
        cleaned = np.zeros_like(boundaries)
        last_b = -min_gap
        for t in range(len(boundaries)):
            if boundaries[t] > 0 and (t - last_b) >= min_gap:
                cleaned[t] = 1.0
                last_b = t
        all_boundaries.append(cleaned)

    return all_boundaries

approx_boundaries = derive_approximate_boundaries(episodes, cfg)
avg_b = np.mean([b.sum() for b in approx_boundaries])
print(f"Approximate boundaries per episode: {avg_b:.1f}")

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
idx = 0
ep0 = episodes[idx][:len(approx_boundaries[idx])]
axes[0].plot(ep0[:, :7], alpha=0.6)
axes[0].set_ylabel('Joints')
axes[0].set_title(f'Episode {idx}: Actions + Approximate Boundaries')
axes[1].plot(ep0[:, 7], 'k-', linewidth=2)
axes[1].set_ylabel('Gripper')
axes[2].stem(approx_boundaries[idx], linefmt='r-', markerfmt='ro', basefmt='gray')
axes[2].set_ylabel('Boundary')
axes[2].set_xlabel('Timestep')
plt.tight_layout()
plt.show()

## 7. Prepare Training Data

Create fixed-length subsequences from residual streams for batch training.

In [ ]:
def create_training_data(residual_streams, episodes, approx_boundaries, cfg):
    """Create fixed-length subsequences for MetaController training."""
    seq_len = cfg.subsequence_len
    horizon = cfg.action_horizon

    all_residuals = []  # (N, seq_len, width)
    all_actions = []    # (N, seq_len, action_dim)
    all_bounds = []     # (N, seq_len)

    for ep_idx in range(len(residual_streams)):
        rs = residual_streams[ep_idx]
        ep = episodes[ep_idx]
        bd = approx_boundaries[ep_idx]
        T = len(rs)

        # Extract overlapping subsequences
        stride = seq_len // 2  # 50% overlap
        for start in range(0, T - seq_len + 1, stride):
            end = start + seq_len
            all_residuals.append(rs[start:end])
            all_actions.append(ep[start:end, :cfg.action_dim])
            all_bounds.append(bd[start:end])

    residuals_tensor = torch.from_numpy(np.array(all_residuals, dtype=np.float32))
    actions_tensor = torch.from_numpy(np.array(all_actions, dtype=np.float32))
    bounds_tensor = torch.from_numpy(np.array(all_bounds, dtype=np.float32))

    print(f"Training subsequences: {len(residuals_tensor)}")
    print(f"  Residuals: {residuals_tensor.shape}")
    print(f"  Actions: {actions_tensor.shape}")
    print(f"  Boundaries: {bounds_tensor.shape}")
    return residuals_tensor, actions_tensor, bounds_tensor

res_data, act_data, bnd_data = create_training_data(
    residual_streams, episodes, approx_boundaries, cfg
)

## 8. MetaController

Same architecture as the paper:
- **Encoder** (BiGRU): non-causal, processes full sequence
- **Switching Unit**: β_t ∈ [0,1] — switch to new controller or maintain current
- **Decoder** (Hypernetwork): z_t → low-rank U_t = B_t @ A_t

In [ ]:
class MetaControllerEncoder(nn.Module):
    """BiGRU encoder (non-causal)."""
    def __init__(self, n_e, n_z, hidden):
        super().__init__()
        self.bigru = nn.GRU(n_e, hidden, bidirectional=True, batch_first=True)
        self.mu_head = nn.Linear(hidden * 2, n_z)
        self.logvar_head = nn.Linear(hidden * 2, n_z)

    def forward(self, e_seq):
        h_seq, _ = self.bigru(e_seq)
        return self.mu_head(h_seq), self.logvar_head(h_seq), h_seq


class SwitchingUnit(nn.Module):
    def __init__(self, n_e, h_dim, n_z):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_e + h_dim + n_z, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, e_t, h_t, z_prev):
        return self.net(torch.cat([e_t, h_t, z_prev], dim=-1)).squeeze(-1)


class ControllerDecoder(nn.Module):
    """Hypernetwork: z_t -> low-rank U_t = B_t @ A_t."""
    def __init__(self, n_z, n_e, rank):
        super().__init__()
        self.n_e = n_e
        self.rank = rank
        self.proj_B = nn.Linear(n_z, n_e * rank)
        self.proj_A = nn.Linear(n_z, rank * n_e)

    def apply_control(self, e_t, z_t):
        """e'_t = e_t + B_t @ (A_t @ e_t)"""
        B_mat = self.proj_B(z_t).view(-1, self.n_e, self.rank)
        A_mat = self.proj_A(z_t).view(-1, self.rank, self.n_e)
        Ae = torch.einsum("bri,bi->br", A_mat, e_t)
        BAe = torch.einsum("bor,br->bo", B_mat, Ae)
        return e_t + BAe


class MetaController(nn.Module):
    def __init__(self, n_e, n_z, rank, encoder_hidden):
        super().__init__()
        self.n_z = n_z
        self.encoder = MetaControllerEncoder(n_e, n_z, encoder_hidden)
        self.switch = SwitchingUnit(n_e, encoder_hidden * 2, n_z)
        self.decoder = ControllerDecoder(n_z, n_e, rank)

    def forward(self, e_seq):
        B, T, _ = e_seq.shape
        mu, logvar, h_seq = self.encoder(e_seq)
        z_list, beta_list, kl_list = [], [], []
        z_prev = torch.zeros(B, self.n_z, device=e_seq.device)

        for t in range(T):
            std = torch.exp(0.5 * logvar[:, t])
            z_tilde = mu[:, t] + std * torch.randn_like(std)
            kl = -0.5 * (1 + logvar[:, t] - mu[:, t].pow(2) - logvar[:, t].exp())
            beta = self.switch(e_seq[:, t], h_seq[:, t], z_prev)
            z_t = beta.unsqueeze(-1) * z_tilde + (1 - beta.unsqueeze(-1)) * z_prev
            z_list.append(z_t)
            beta_list.append(beta)
            kl_list.append(kl.sum(dim=-1))
            z_prev = z_t

        z_seq = torch.stack(z_list, dim=1)
        beta_seq = torch.stack(beta_list, dim=1)
        kl_loss = torch.stack(kl_list, dim=1).mean()
        return z_seq, kl_loss, beta_seq

meta = MetaController(
    n_e=cfg.width, n_z=cfg.n_z, rank=cfg.rank, encoder_hidden=cfg.encoder_hidden
).to(device)
n_meta = sum(p.numel() for p in meta.parameters())
print(f"MetaController: {n_meta / 1e6:.2f}M parameters")
print(f"  Encoder (BiGRU): {sum(p.numel() for p in meta.encoder.parameters()) / 1e3:.1f}K")
print(f"  Switching Unit:  {sum(p.numel() for p in meta.switch.parameters()) / 1e3:.1f}K")
print(f"  Decoder:         {sum(p.numel() for p in meta.decoder.parameters()) / 1e6:.2f}M")

## 9. Expert Distill Training

Train MetaController on frozen expert's residual stream.

**Loss**: `L(φ) = reconstruction + α * KL`
- Reconstruction: can the controlled residual stream reconstruct the original actions?
- KL: rate-distortion tradeoff (controls boundary granularity)

In [ ]:
# Action decoder: controlled residual -> action prediction
action_decoder = nn.Linear(cfg.width, cfg.action_dim).to(device)

optimizer = torch.optim.AdamW(
    list(meta.parameters()) + list(action_decoder.parameters()),
    lr=cfg.distill_lr, weight_decay=0.01,
)

meta.train()
action_decoder.train()

history = {"total": [], "recon": [], "kl": [], "beta_mean": [], "beta_sparsity": []}
start = time.time()
N = len(res_data)

for step in range(cfg.distill_steps):
    idx = torch.randint(0, N, (cfg.distill_batch_size,))
    e_seq = res_data[idx].to(device)
    target = act_data[idx].to(device)

    z_seq, kl_loss, beta_seq = meta(e_seq)

    B, T, n_e = e_seq.shape
    e_flat = e_seq.reshape(B * T, n_e)
    z_flat = z_seq.reshape(B * T, cfg.n_z)
    e_controlled = meta.decoder.apply_control(e_flat, z_flat)
    pred = action_decoder(e_controlled).reshape(B, T, cfg.action_dim)

    recon_loss = F.mse_loss(pred, target)
    total_loss = recon_loss + cfg.alpha * kl_loss

    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(meta.parameters(), 1.0)
    optimizer.step()

    history["total"].append(total_loss.item())
    history["recon"].append(recon_loss.item())
    history["kl"].append(kl_loss.item())
    history["beta_mean"].append(beta_seq.mean().item())
    history["beta_sparsity"].append((beta_seq > 0.5).float().mean().item())

    if (step + 1) % 500 == 0:
        print(f"Step {step+1}/{cfg.distill_steps} | "
              f"Total: {total_loss.item():.4f} | Recon: {recon_loss.item():.4f} | "
              f"KL: {kl_loss.item():.4f} | beta_mean: {beta_seq.mean().item():.3f} | "
              f"beta>0.5: {(beta_seq > 0.5).float().mean().item():.3f} | "
              f"{time.time()-start:.0f}s")

print(f"\nExpert Distill complete in {time.time()-start:.0f}s")

In [ ]:
# Training curves
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
smooth = lambda x, w=50: np.convolve(x, np.ones(w)/w, mode='valid')

for ax, key, title in [
    (axes[0,0], 'total', 'Total Loss'),
    (axes[0,1], 'recon', 'Reconstruction Loss'),
    (axes[1,0], 'kl', 'KL Divergence'),
    (axes[1,1], 'beta_sparsity', 'Beta Sparsity (frac > 0.5)'),
]:
    ax.plot(history[key], alpha=0.3, color='blue')
    if len(history[key]) > 50:
        ax.plot(smooth(history[key]), 'r-', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Step')
    ax.grid(True, alpha=0.3)

if 'beta_sparsity' in [a[1] for a in [(axes[1,1], 'beta_sparsity')]]:
    axes[1,1].set_ylim(0, 1)

plt.suptitle('Expert Distill Training (Real pi0.5-DROID)', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Evaluate: Does β_t Discover Boundaries?

The key question: **without any boundary supervision, does β_t fire at meaningful subtask transitions in the real DROID data?**

In [ ]:
def evaluate_beta_patterns(meta, res_data, act_data, bnd_data, num_examples=5):
    """Visualize beta_t vs approximate boundaries."""
    meta.eval()
    with torch.no_grad():
        e_seq = res_data[:num_examples].to(device)
        _, _, beta_seq = meta(e_seq)
        beta_np = beta_seq.cpu().numpy()

    fig, axes = plt.subplots(num_examples, 1, figsize=(14, 3 * num_examples), sharex=True)
    if num_examples == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        t = np.arange(cfg.subsequence_len)
        ax.plot(t, beta_np[i], 'b-', linewidth=1.5, alpha=0.8, label='Learned beta_t')
        ax.fill_between(t, 0, beta_np[i], alpha=0.2, color='blue')

        # Approximate boundaries (red dashed)
        gt = bnd_data[i].numpy()
        for idx in np.where(gt > 0.5)[0]:
            ax.axvline(x=idx, color='red', linestyle='--', alpha=0.7, linewidth=2)

        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
        ax.set_ylabel(f'Seq {i}')
        ax.set_ylim(-0.05, 1.05)
        if i == 0:
            ax.legend(loc='upper right')
        ax.grid(True, alpha=0.2)

    axes[0].set_title(
        'Learned beta_t (blue) vs Approximate Boundaries (red dashed)\n'
        'beta_t > 0.5 = subtask boundary detected', fontsize=12
    )
    axes[-1].set_xlabel('Timestep')
    plt.tight_layout()
    plt.show()

evaluate_beta_patterns(meta, res_data, act_data, bnd_data, num_examples=5)

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

def compute_metrics(meta, res_data, bnd_data, cfg, tolerance=3):
    """Compute boundary detection metrics."""
    meta.eval()
    all_nmi, all_f1, all_tcr = [], [], []

    with torch.no_grad():
        for i in range(0, len(res_data), 64):
            e_seq = res_data[i:i+64].to(device)
            gt_batch = bnd_data[i:i+64]
            _, _, beta_seq = meta(e_seq)
            beta_np = beta_seq.cpu().numpy()
            gt_np = gt_batch.numpy()

            for j in range(len(beta_np)):
                pred_b = (beta_np[j] > 0.5).astype(float)
                gt_b = (gt_np[j] > 0.5).astype(float)

                # NMI
                nmi = normalized_mutual_info_score(gt_b, pred_b)
                all_nmi.append(nmi)

                # F1 with tolerance
                pred_idx = set(np.where(pred_b > 0.5)[0])
                gt_idx = set(np.where(gt_b > 0.5)[0])
                tp = sum(1 for p in pred_idx if any(abs(p - g) <= tolerance for g in gt_idx))
                prec = tp / max(len(pred_idx), 1)
                rec = tp / max(len(gt_idx), 1)
                f1 = 2 * prec * rec / max(prec + rec, 1e-8)
                all_f1.append(f1)

                # TCR (temporal contraction ratio)
                n_sw = max(int(pred_b.sum()), 1)
                all_tcr.append(len(pred_b) / n_sw)

    print("=" * 50)
    print("Expert Distill Evaluation (Real pi0.5-DROID)")
    print("=" * 50)
    print(f"Switching NMI:         {np.mean(all_nmi):.4f} +/- {np.std(all_nmi):.4f}")
    print(f"Boundary F1 (tol={tolerance}):  {np.mean(all_f1):.4f} +/- {np.std(all_f1):.4f}")
    print(f"Temporal Contraction:   {np.mean(all_tcr):.1f}x +/- {np.std(all_tcr):.1f}x")
    print(f"Avg switches/seq:       {cfg.subsequence_len / np.mean(all_tcr):.1f}")
    print(f"Avg GT boundaries/seq:  {bnd_data.sum(dim=1).mean():.1f}")

compute_metrics(meta, res_data, bnd_data, cfg)

In [ ]:
# Beta distribution: check quasi-binary property
meta.eval()
with torch.no_grad():
    e_seq = res_data[:min(200, len(res_data))].to(device)
    _, _, beta_seq = meta(e_seq)
    all_betas = beta_seq.cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_betas, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', label='threshold=0.5')
axes[0].set_xlabel('beta_t')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of beta_t')
axes[0].legend()

near_0 = (all_betas < 0.1).mean() * 100
near_1 = (all_betas > 0.9).mean() * 100
middle = ((all_betas >= 0.1) & (all_betas <= 0.9)).mean() * 100

axes[1].bar(['near 0\n(< 0.1)', 'middle\n(0.1-0.9)', 'near 1\n(> 0.9)'],
            [near_0, middle, near_1],
            color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='black')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Quasi-binary check (ideal: mass at 0 and 1)')

plt.suptitle('Beta_t Analysis', fontsize=13)
plt.tight_layout()
plt.show()

print(f"near 0 (<0.1): {near_0:.1f}%")
print(f"middle (0.1-0.9): {middle:.1f}%")
print(f"near 1 (>0.9): {near_1:.1f}%")
print(f"Quasi-binary: {'YES' if middle < 30 else 'NOT YET (try more training or different alpha)'}")

## Summary

**What this notebook tested:**
1. Built Gemma-300M with **adaRMSNorm** (matching real π0.5 architecture)
2. Loaded **pretrained weights** from `pi05_droid` JAX checkpoint
3. Froze expert and extracted **residual streams** at layer 9 from real DROID data
4. Trained MetaController (BiGRU + SwitchingUnit + Hypernetwork) on frozen residuals
5. Evaluated whether β_t discovers subtask boundaries **without supervision**

**Key observations:**
- β_t should show **quasi-binary** pattern (values near 0 or 1)
- Boundary firing should correlate with action velocity changes and gripper transitions
- α controls boundary granularity (larger α = coarser boundaries)

**Next steps:**
- Scale to full DROID dataset (not just droid_100)
- Phase 3: Internal RL for task-success optimization
- Deploy with VLM planner for capability-aware subtask decomposition